In [14]:
import cv2
import os
from collections import defaultdict

# Input and output paths
input_dir = r"../downloaded_videos"
output_dir = r"standardized_videos"

# Check if input directory exists
if not os.path.exists(input_dir):
    print(f"❌ Input directory not found: {os.path.abspath(input_dir)}")
    exit()
# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
# Initialize variables for lowest resolution and FPS
min_width, min_height, min_fps = float("inf"), float("inf"), float("inf")
# Step 1: Analyze all videos to find the lowest resolution and FPS
for filename in os.listdir(input_dir):
    if filename.endswith(".mp4"):
        input_path = os.path.join(input_dir, filename)
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"❌ Error opening {filename}")
            continue
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        # Update the minimum values
        min_width = min(min_width, width)
        min_height = min(min_height, height)
        min_fps = min(min_fps, fps)

        cap.release()

# If no valid videos were found
if min_width == float("inf") or min_height == float("inf") or min_fps == float("inf"):
    print("❌ No valid videos found for analysis.")
    exit()

print(f"Lowest Resolution: {min_width}x{min_height}, Lowest FPS: {int(min_fps)}")
# Step 2: Group videos into clusters based on a pattern (e.g., folder name or prefix)
clusters = defaultdict(list)
for filename in os.listdir(input_dir):
    if filename.endswith(".mp4"):
        cluster_key = filename.split("_")[1]  # Group by prefix (you can customize this pattern)
        clusters[cluster_key].append(filename)
# Step 3: Standardize videos, taking only 50 videos from each cluster
for cluster_key, filenames in clusters.items():
    print(f"Processing cluster: {cluster_key}")
    # Ensure the cluster contains videos
    if len(filenames) == 0:
        print(f"⚠️ No videos found in cluster: {cluster_key}")
        continue
    # Always take the first 50 videos from each cluster (or less if there aren't 50 videos)
    filenames_to_process = filenames[:50]
    for filename in filenames_to_process:
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        # Print the original size
        original_size = os.path.getsize(input_path) / (1024 * 1024)  # Convert to MB
        print(f"📁 {filename} - Original Size: {original_size:.2f} MB")
        # Open video file
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"❌ Error opening {filename}")
            continue
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")

        out = cv2.VideoWriter(output_path, fourcc,min_fps, (min_width, min_height))
        print(f"🔄 Standardizing {filename}...")
        while True:
            ret, frame = cap.read()
            if not ret:
                break  # End of video
            # Resize frame to lowest resolution
            frame = cv2.resize(frame, (min_width, min_height))
            # Write standardized frame
            out.write(frame)
        # Release resources
        cap.release()
        out.release()
        # Print the standardized size
        standardized_size = os.path.getsize(output_path) / (1024 * 1024)  # Convert to MB
        print(f"✅ Standardized: {filename} - New Size: {standardized_size:.2f} MB")
        print(f"📉 Size Reduction: {original_size - standardized_size:.2f} MB\n")
print("🎉 Standardization complete for all videos!")

Lowest Resolution: 420x316, Lowest FPS: 23
Processing cluster: 0
📁 cluster_0_video_1007789530.mp4 - Original Size: 0.98 MB
🔄 Standardizing cluster_0_video_1007789530.mp4...
✅ Standardized: cluster_0_video_1007789530.mp4 - New Size: 3.22 MB
📉 Size Reduction: -2.24 MB

📁 cluster_0_video_1008025297.mp4 - Original Size: 1.56 MB
🔄 Standardizing cluster_0_video_1008025297.mp4...
✅ Standardized: cluster_0_video_1008025297.mp4 - New Size: 5.29 MB
📉 Size Reduction: -3.73 MB

📁 cluster_0_video_1008225385.mp4 - Original Size: 1.17 MB
🔄 Standardizing cluster_0_video_1008225385.mp4...
✅ Standardized: cluster_0_video_1008225385.mp4 - New Size: 3.80 MB
📉 Size Reduction: -2.63 MB

📁 cluster_0_video_1008352831.mp4 - Original Size: 3.80 MB
🔄 Standardizing cluster_0_video_1008352831.mp4...
✅ Standardized: cluster_0_video_1008352831.mp4 - New Size: 3.83 MB
📉 Size Reduction: -0.03 MB

📁 cluster_0_video_1008400516.mp4 - Original Size: 1.09 MB
🔄 Standardizing cluster_0_video_1008400516.mp4...
✅ Standardized:

In [5]:
import cv2
import os
# Input and output paths
input_dir = r"standardized_videos"
output_dir = r"extracted_standardized_frames"
# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)
# Resolution to standardize the frames (from previous analysis of video resolutions)
min_width =426 
min_height = 316 
# FPS and frame interval for extraction
fps = 23  
interval_seconds = 10
frame_interval = int(fps * interval_seconds) 
# Function to extract and standardize frames at regular intervals (every N seconds)
def extract_and_standardize_frames(video_path, output_folder, interval_seconds=2):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ Error opening video: {video_path}")
        return  
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Extracting and standardizing frames from {video_path} | Total Frames: {total_frames}")  
    frame_count = 0
    saved_frame_count = 0  
    while True:
        ret, frame = cap.read()
        if not ret:
            break  # End of video       
        frame_count += 1       
        # Save frame at regular intervals
        if frame_count % frame_interval == 0:
            # Resize the frame to the standard resolution
            frame_resized = cv2.resize(frame, (min_width, min_height))          
            saved_frame_count += 1
            frame_filename = os.path.join(output_folder, f"{os.path.basename(video_path)}_frame_{saved_frame_count:04d}.png")
            cv2.imwrite(frame_filename, frame_resized)
            print(f"📸 Extracted and standardized frame {saved_frame_count}/{total_frames} from {video_path}")
    cap.release()
    print(f"✅ Finished extracting and standardizing frames from {video_path}.\n")
# Extract and standardize frames from all standardized videos
for filename in os.listdir(input_dir):
    if filename.endswith(".mp4"):
        video_path = os.path.join(input_dir, filename)
        extract_and_standardize_frames(video_path, output_dir, interval_seconds=interval_seconds)
print("🎉 Frame extraction and standardization complete!")

Extracting and standardizing frames from standardized_videos\cluster_0_video_1007789530.mp4 | Total Frames: 228
✅ Finished extracting and standardizing frames from standardized_videos\cluster_0_video_1007789530.mp4.

Extracting and standardizing frames from standardized_videos\cluster_0_video_1008025297.mp4 | Total Frames: 356
📸 Extracted and standardized frame 1/356 from standardized_videos\cluster_0_video_1008025297.mp4
✅ Finished extracting and standardizing frames from standardized_videos\cluster_0_video_1008025297.mp4.

Extracting and standardizing frames from standardized_videos\cluster_0_video_1008225385.mp4 | Total Frames: 300
📸 Extracted and standardized frame 1/300 from standardized_videos\cluster_0_video_1008225385.mp4
✅ Finished extracting and standardizing frames from standardized_videos\cluster_0_video_1008225385.mp4.

Extracting and standardizing frames from standardized_videos\cluster_0_video_1008352831.mp4 | Total Frames: 879
📸 Extracted and standardized frame 1/879 fr

Object Detection using YOLO and saving to csv file with mapped Video Id

In [ ]:
import torch
from PIL import Image
import os
import shutil
import cv2
import re

# Load YOLOv5 model
print("🚀 Loading YOLOv5 model...")
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)  # Or your custom model

# Directories (Adjust these paths)
frame_dir = r"extracted_standardized_frames"  # Input folder containing frames
output_detection_dir = r"detected_frames"  # Main folder for detected frames (organized by video)
standardized_videos_dir = r"standardized_videos"  # Directory containing standardized videos

os.makedirs(output_detection_dir, exist_ok=True)

def extract_video_id_and_cluster(frame_filename):
    """Extracts video ID and cluster from the frame filename using regex."""
    match = re.search(r"cluster_(\d+)_video_(\d+)", frame_filename)
    if match:
        return match.group(1), match.group(2)  # Return cluster and video ID
    return None, None


def perform_object_detection(frame_path, output_video_dir, frame_filename):
    img = Image.open(frame_path)
    results = model(img)

    detected_image_name = f"detected_{frame_filename}"
    output_path = os.path.join(output_video_dir, detected_image_name)

    annotated_image = results.render()[0]
    annotated_image_bgr = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_path, annotated_image_bgr)

    return results
print("🔍 Starting object detection and video copying...")

for frame_filename in os.listdir(frame_dir):
    if frame_filename.endswith((".png", ".jpg", ".jpeg")):
        frame_path = os.path.join(frame_dir, frame_filename)
        print(f"➡️ Processing {frame_filename}")

        video_id = extract_video_id_and_cluster(frame_filename)

        if video_id:
            # ***CORRECT MATCHING LOGIC:***  Use only the video_id to find the video
            video_filename = None  # Initialize to None
            for video_file in os.listdir(standardized_videos_dir):
                if f"cluster_9_video_{video_id}.mp4" == video_file:  # Exact match
                    video_filename = video_file
                    break  # Stop searching once found

            if video_filename:  # If the video filename exists
                source_video_path = os.path.join(standardized_videos_dir, video_filename)

                output_video_dir = os.path.join(output_detection_dir, f"video_{video_id}")
                os.makedirs(output_video_dir, exist_ok=True)

                perform_object_detection(frame_path, output_video_dir, frame_filename)

                destination_video_path = os.path.join(output_video_dir, video_filename)
                if os.path.exists(source_video_path):
                    shutil.copy2(source_video_path, destination_video_path)
                    print(f"Copied video {video_filename} to {destination_video_path}")
                else:
                    print(f"⚠️ Video {video_filename} not found in standardized_videos!")
            else:
                print(f"⚠️ Video for ID {video_id} not found in standardized_videos!")
        else:
            print(f"⚠️ Could not extract video ID from frame filename: {frame_filename}")

print("🎉 Object detection and video copying completed!")

🚀 Loading YOLOv5 model...


Using cache found in C:\Users\harpr/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2025-2-10 Python-3.12.7 torch-2.6.0+cu118 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


➡️ Processing cluster_0_video_1008025297.mp4_frame_0001.png
➡️ Processing cluster_0_video_1008225385.mp4_frame_0001.png
➡️ Processing cluster_0_video_1008352831.mp4_frame_0001.png
➡️ Processing cluster_0_video_1008352831.mp4_frame_0002.png
➡️ Processing cluster_0_video_1008352831.mp4_frame_0003.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0001.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0002.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0003.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0004.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0005.png
➡️ Processing cluster_0_video_1009213511.mp4_frame_0006.png
➡️ Processing cluster_0_video_1009911737.mp4_frame_0001.png
➡️ Processing cluster_0_video_1009911737.mp4_frame_0002.png
➡️ Processing cluster_0_video_1009911737.mp4_frame_0003.png
➡️ Processing cluster_0_video_1009911737.mp4_frame_0004.png
➡️ Processing cluster_0_video_1010861270.mp4_frame_0001.png
➡️ Processing cluster_0_video_1011464780

In [8]:
import pandas as pd
df=pd.read_csv('video_detections_summary.csv')
df.head(5)

,video_id,detected_objects
0,1008025297.mp4,person
1,1008225385.mp4,person
2,1008352831.mp4,bird
3,1009213511.mp4,"person, backpack, handbag"
4,1011708968.mp4,cell phone
